In [1]:
import torch

In [2]:
CONTEXT_LENGTH = 768
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
@torch.no_grad()
def generate(model,tokenizer, prompt, max_new_tokens=100, temperature=1.0):
    model.eval()

    tokens = tokenizer.encode(prompt)
    x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):

        # Keep only the most recent context
        x_cond = x[:, -CONTEXT_LENGTH:]
        # print("printed : ",x_cond)


        logits = model(x_cond)

        # Take logits for the final position
        logits = logits[:, -1, :] / temperature

        probs = torch.softmax(logits, dim=-1)

        # Sample next token
        next_token = torch.multinomial(probs, num_samples=1)

        x = torch.cat((x, next_token), dim=1)

    generated_tokens = x[0].tolist()

    return tokenizer.decode(generated_tokens)

In [5]:
state_dict = torch.load("latest_checkpoint_1.pt",map_location=device)

In [6]:
from ddp_2 import Transformer

In [7]:
VOCAB_SIZE = 50257
CONTEXT_LENGTH = 768
STRIDE = 768
BATCH_SIZE = 8
EMBEDDED_DIM = 768
NUM_LAYERS = 8
NUM_HEADS = 8
DROPOUT = 0.1
EPOCHS = 1
ACCUMULATION_STEPS = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1

In [8]:
model = Transformer(
        emb_dim=EMBEDDED_DIM,
        num_heads=NUM_HEADS,
        dropout=DROPOUT,
        num_layers=NUM_LAYERS,
        is_causal=True,
        context_length=CONTEXT_LENGTH,
        vocab_size=VOCAB_SIZE
    ).to(device)

In [9]:
print(generate(model,tokenizer,"Physics is",20,0.7))

Physics is015 II reasonsecution hypert Features Obesity effortlesslyutererers PCIe Lent timers Royal arrivesorous propheticyeah Giants restroom


In [12]:
model.load_state_dict(state_dict=state_dict["model"])
print(generate(model,tokenizer,"Love is",40,0.7))

Love is the first thing we talked about is the Bizanko and the second thing we discussed in the first week is, it is the idea that it is the last thing we talked about was the B


In [13]:
state_dict_2 = torch.load("latest_checkpoint_2.pt",map_location=device)
model.load_state_dict(state_dict=state_dict_2["model"])

<All keys matched successfully>

In [14]:
print(generate(model,tokenizer,"Love is",40,0.8))

Love is the thing only to do.
I entered the program last week, since I couldn't befied, so I'm sure it was silver time.
So, I don't think the program


In [15]:
del state_dict
del state_dict_2

In [17]:
state_dict_3 = torch.load("latest_checkpoint_3.pt",map_location=device)
model.load_state_dict(state_dict=state_dict_3["model"])

<All keys matched successfully>

In [18]:
print(generate(model,tokenizer,"Love is",40,0.8))
del state_dict_3

Love is the first one to fall in love with this eighth grade post.
Hey there! I love it!
The school of the gif works in a way to tell the story of the school and the


In [42]:
best_state_dict = torch.load("best_model_checkpoint.pt",map_location=device)
model.load_state_dict(state_dict=best_state_dict["model"])

<All keys matched successfully>

In [49]:
for i,prompts in enumerate([
    "The Lagrangian of a free real scalar field is",
    "The Klein-Gordon equation is obtained from",
    "In special relativity, the invariant interval is",
    "The Schrödinger equation for a free particle is",
    "The Standard Model gauge group is",
]):
    

    print(f"Prompt {i+1} : {generate(model,tokenizer,prompts,60,0.8)}\n\n")


Prompt 1 : The Lagrangian of a free real scalar field is the most painful scene in modern history. The photos are incredible. The Lagrangian is old and very funny. The rivers are almost out of control. The Lagrangians are not equipped to play the games. They have to play well. They have to play well, play well. The Lag


Prompt 2 : The Klein-Gordon equation is obtained from a barcode or a Q&A session. Here is a sample of the equations:
- The mean values of these weights are multiplying in the low-intensity barcode area.
- The mean values of these weights are increasing.
- The mean values of the points of each weights are


Prompt 3 : In special relativity, the invariant interval is the time for an enumerating interval as much as 1,000 milliseconds. This pattern is usually used when calculating a measurement of the time of Tsunami (E, Kb, L, etc.) . In one study, a little of the time of Tsunami in the Tyla


Prompt 4 : The Schrödinger equation for a free particle is that of the fluid co

In [2]:
import subprocess

pdf_path = "/home/tuhin/Downloads/Atomic_Physics_Christopher_Foot.pdf"
output_txt_path = "output.txt"

# Run the native OS command: pdftotext input.pdf output.txt
try:
    subprocess.run(["pdftotext", pdf_path, output_txt_path], check=True)
    print(f"Success! Text extracted natively to {output_txt_path}")
except FileNotFoundError:
    print("The 'pdftotext' command-line utility is not installed on your system.")


Success! Text extracted natively to output.txt
